# Apache Spark Assignment – Week 6

In [1]:
import os
import sys

os.environ["PYSPARK_DRIVER_PYTHON"]= sys.executable
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ['HADOOP_HOME'] = "C:\\hadoop"
os.environ['PATH'] = os.environ['PATH'] + ";C:\\hadoop\\bin"

# Q1. Explain the roles of the Driver, Cluster Manager, and Executor in a Spark application.

## Introduction

Apache Spark follows a master-worker architecture to process large amounts of data in a distributed environment. Every Spark application consists of three main components:

- Driver
- Cluster Manager
- Executors

Each component has a specific responsibility, and together they enable Spark to process data efficiently across multiple machines.

## 1. Driver

The **Driver** is the main program that controls the Spark application.

**Responsibilities:**
- Creates the `SparkSession`.
- Converts the program into tasks.
- Sends tasks to Executors.
- Collects the final results.

### Example

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark = SparkSession.builder.appName("SalesAnalysis").getOrCreate()

Here, the Driver creates the SparkSession and prepares the execution plan for reading and processing the data.

## 2. Cluster Manager

The **Cluster Manager** manages the resources of the cluster.

**Responsibilities:**
- Allocates CPU and memory.
- Starts Executors on worker nodes.
- Manages resources for Spark applications.

**Examples:** Standalone, Hadoop YARN, Kubernetes, Apache Mesos.

## 3. Executor

An **Executor** is a worker process that performs the actual computation.

**Responsibilities:**
- Executes tasks assigned by the Driver.
- Processes data partitions.
- Stores intermediate data in memory.
- Returns the results to the Driver.

### Example

Suppose a CSV file is divided into **4 partitions**. Spark can assign each partition to a different Executor, allowing all four partitions to be processed simultaneously. This parallel processing makes Spark much faster.

## Working Flow

1. The Driver starts the application.
2. The Cluster Manager allocates resources.
3. Executors are launched on worker nodes.
4. Executors process the data in parallel.
5. Results are returned to the Driver.

## Architecture Diagram

```

           Driver
              │
      Creates Tasks
              │
              ▼
      Cluster Manager
              │
 Allocates Resources
              │
   ┌──────────┼──────────┐
   ▼          ▼          ▼
Executor 1  Executor 2  Executor 3
(Process)   (Process)   (Process)
   │          │          │
   └──────────┼──────────┘
              │
      Returns Results
              │
              ▼
           Driver

```

# Q2. How does Spark’s Lazy Evaluation strategy improve performance when chain-processing large datasets?

## What is Lazy Evaluation?

Lazy Evaluation is a feature of Apache Spark where transformations are **not executed immediately**. Instead, Spark records all the transformations and waits until an **action** is called. It then creates the most efficient execution plan and executes all operations together.

## How It Improves Performance

- Reduces unnecessary computations.
- Optimizes the execution plan using the Catalyst Optimizer.
- Minimizes data movement (shuffle) whenever possible.
- Executes multiple transformations in a single optimized job.
- Saves memory and improves processing speed for large datasets.

## How It Improves Performance

- Reduces unnecessary computations.
- Optimizes the execution plan using the Catalyst Optimizer.
- Minimizes data movement (shuffle) whenever possible.
- Executes multiple transformations in a single optimized job.
- Saves memory and improves processing speed for large datasets.

## Example

In [3]:
df = spark.read.csv(
    "data/source.csv",
    header=True,
    inferSchema=True
)
result = (df.filter(df["status"] == "Completed")
            .filter(df["discount_percent"] > 10)
            .select("product_name", "region", "price")
            .groupBy("region")
            .avg("price"))

result.show()

+------+-----------------+
|region|       avg(price)|
+------+-----------------+
| South|52511.73743589744|
|  East|46587.68191489359|
|  West|44750.21050000002|
| North| 53106.6615151515|
+------+-----------------+



- `filter()`, `select()`, and `groupBy()` are **transformations**.
- Spark does **not execute** these transformations immediately.
- It waits until `show()` (an **action**) is called.
- When `show()` is executed, Spark combines all the transformations into a single optimized execution plan instead of running them one by one. This reduces unnecessary computations and improves performance while processing large datasets.

# Q3. Write a Spark command to read a CSV file located at `"data/source.csv"`, ensuring the first row is treated as a header and `inferSchema` is enabled.

In [4]:
# Read the CSV file with header and automatic schema inference

df = spark.read.csv(
    "data/source.csv",
    header=True,
    inferSchema=True
)

# Display the first 5 rows
df.show(5)

+--------+----------+------------+-----------+--------+--------+----------------+------+---------+-------------+----------+
|order_id|product_id|product_name|   category|   price|quantity|discount_percent|region|   status|customer_name|order_date|
+--------+----------+------------+-----------+--------+--------+----------------+------+---------+-------------+----------+
|  100001|      1001|      Laptop|Electronics|74180.89|       4|              20| North| Returned|  Customer_17|31-01-2025|
|  100002|      NULL|     Monitor|Electronics|23342.82|      10|              20|  West|  Pending| Customer_230|12-10-2025|
|  100003|      1003|    Keyboard|Electronics|69844.13|       6|              10| North|Completed| Customer_195|10-04-2025|
|  100004|      1004|      Jacket|   Clothing|60412.23|       1|              15| North|Cancelled| Customer_425|06-01-2026|
|  100005|      1005|    Football|     Sports|70486.73|       1|               0| South|Completed| Customer_195|12-10-2025|
+-------

# Q4. What is the difference between CSV and Parquet in terms of storage (row-based vs. columnar) and why does it matter for performance?

## CSV vs Parquet

CSV (Comma-Separated Values) is a **row-based** file format where all the values of a single row are stored together. It is simple, human-readable, and supported by almost every data processing tool. However, it does not store metadata such as data types, so Spark often needs to infer the schema while reading the file.

Parquet is a **columnar** file format where values of the same column are stored together. It is optimized for big data processing frameworks like Apache Spark. Parquet stores metadata, supports compression, and allows Spark to read only the required columns, making it much faster than CSV for analytical workloads.

## Difference Between CSV and Parquet

| Feature | CSV | Parquet |
|---------|-----|----------|
| Storage Format | Row-based | Columnar |
| Readability | Human-readable | Binary format (not human-readable) |
| Schema | Does not store schema | Stores schema and metadata |
| Compression | Usually not compressed | Supports efficient compression |
| File Size | Larger | Smaller |
| Query Performance | Slower | Faster |
| Best Use | Data exchange and simple storage | Big data analytics and Spark processing |

## Why Columnar Storage Improves Performance

In a row-based format like CSV, Spark reads every column in each row even if only a few columns are required. This increases disk I/O and processing time.

In a columnar format like Parquet, Spark reads only the required columns. Since less data is read from disk, queries execute much faster. Parquet also compresses similar data values in each column, reducing storage space and improving read performance.

## Example

Suppose the dataset contains the following columns:

`order_id`, `product_id`, `product_name`, `category`, `price`, `quantity`, `discount_percent`, `region`, `status`, `customer_name`, `order_date`

Now, I want to display only the `product_name` and `price` columns.

```python
df.select("product_name", "price").show()
```

### CSV

Spark reads all 11 columns from every row and then selects only `product_name` and `price`. This increases the amount of data read from disk.

### Parquet

Spark reads only the `product_name` and `price` columns because they are stored separately. This reduces disk I/O, speeds up execution, and uses less memory.

# Q5. Given a DataFrame `df`, write a query to select the columns `product_id` and `price` where the `category` is **'Electronics'**.

In [5]:
df.filter(df["category"] == "Electronics").select("product_id","price").show()

+----------+--------+
|product_id|   price|
+----------+--------+
|      1001|74180.89|
|      NULL|23342.82|
|      1003|69844.13|
|      1014|87884.32|
|      1015|76606.86|
|      1018| 7178.65|
|      1020|19239.85|
|      NULL|21394.12|
|      1029|15629.25|
|      1030|    NULL|
|      1047|23525.91|
|      1056|69092.38|
|      1061|72035.46|
|      1062|22237.48|
|      1068|11933.31|
|      1080|94711.37|
|      1084|74496.04|
|      1086|78174.38|
|      1092|38198.29|
|      1101|61603.66|
+----------+--------+
only showing top 20 rows


# Q6. Write the code to revise a DataFrame by renaming the column `old_name` to `new_name` and casting the `price` column from a String to a Double.

In [6]:
from pyspark.sql.types import DoubleType

# Rename the column and cast price from String to Double
revised_df = df.withColumnRenamed("category", "Product_Category") \
               .withColumn("price", col("price").cast(DoubleType()))

# Display the updated DataFrame
revised_df.show()

revised_df.printSchema()

+--------+----------+----------------+----------------+--------+--------+----------------+------+---------+-------------+----------+
|order_id|product_id|    product_name|Product_Category|   price|quantity|discount_percent|region|   status|customer_name|order_date|
+--------+----------+----------------+----------------+--------+--------+----------------+------+---------+-------------+----------+
|  100001|      1001|          Laptop|     Electronics|74180.89|       4|              20| North| Returned|  Customer_17|31-01-2025|
|  100002|      NULL|         Monitor|     Electronics|23342.82|      10|              20|  West|  Pending| Customer_230|12-10-2025|
|  100003|      1003|        Keyboard|     Electronics|69844.13|       6|              10| North|Completed| Customer_195|10-04-2025|
|  100004|      1004|          Jacket|        Clothing|60412.23|       1|              15| North|Cancelled| Customer_425|06-01-2026|
|  100005|      1005|        Football|          Sports|70486.73|     

# Q7. How does Spark use the Lineage Graph (DAG) to provide fault tolerance if a worker node fails?

## Lineage Graph (DAG) in Spark

A **Lineage Graph**, also called a **Directed Acyclic Graph (DAG)**, is a record of all the transformations performed on a dataset. Instead of storing multiple copies of data, Spark keeps track of how the data was created from the original source.

This lineage information helps Spark recover lost data if a worker node or Executor fails.

## Fault Tolerance Using DAG

When a worker node fails, the data stored in its memory may be lost. Spark does not restart the entire application. Instead, it uses the Lineage Graph (DAG) to identify the lost partition and recomputes only that partition from the original data using the recorded transformations.

This approach:
- Avoids reprocessing the entire dataset.
- Saves time and resources.
- Ensures reliable execution even if failures occur.

## Example

```python
df = spark.read.csv("data/source.csv", header=True, inferSchema=True)

filtered_df = df.filter(df["status"] == "completed")

result = filtered_df.groupBy("region").sum("price")
```

Here, Spark records the following transformations:

1. Read the `orders.csv` file.
2. Filter rows where `status = "Delivered"`.
3. Group the data by `region`.
4. Calculate the sum of `price`.

If an Executor processing one partition fails during the `groupBy()` operation, Spark uses the DAG to recompute only the lost partition instead of executing the entire program again.

# Q8. Write a query to filter a DataFrame `df_orders` for rows where the `status` is **'Completed'** AND the `amount` is greater than **1000**.

In [7]:
df_orders = df.filter(
    (col("status") == "Completed") & (col("price") > 1000)
)
df_orders.show()

+--------+----------+----------------+-----------+--------+--------+----------------+------+---------+-------------+----------+
|order_id|product_id|    product_name|   category|   price|quantity|discount_percent|region|   status|customer_name|order_date|
+--------+----------+----------------+-----------+--------+--------+----------------+------+---------+-------------+----------+
|  100003|      1003|        Keyboard|Electronics|69844.13|       6|              10| North|Completed| Customer_195|10-04-2025|
|  100005|      1005|        Football|     Sports|70486.73|       1|               0| South|Completed| Customer_195|12-10-2025|
|  100007|      1007|            Eggs|  Groceries|98953.38|       9|               0| South|Completed| Customer_413|20-11-2025|
|  100010|      1010|Badminton Racket|     Sports| 9181.85|       1|               5|  West|Completed| Customer_198|26-01-2026|
|  100014|      1014|          Tablet|Electronics|87884.32|       5|              20| North|Completed| C

> **Note:** The question refers to an `amount` column. However, my dataset does not contain an `amount` column, so I have used the `price` column instead to demonstrate the required filtering operation.

# Q9. Explain the concept of Predicate Pushdown in Parquet and how it affects the amount of data loaded into memory.

## What is Predicate Pushdown?

Predicate Pushdown is an optimization technique used by Apache Spark when reading Parquet files. Instead of loading the entire dataset into memory and then applying filters, Spark pushes the filter condition directly to the Parquet file.

As a result, only the rows that satisfy the filter condition are read from disk, while unnecessary data is skipped.

## How It Improves Performance

Predicate Pushdown provides the following benefits:

- Reduces the amount of data read from disk.
- Loads only the required data into memory.
- Decreases disk I/O operations.
- Improves query execution speed.
- Makes Spark applications more efficient when working with large datasets.

## Example

Suppose the dataset contains the columns:

`order_id`, `product_name`, `category`, `price`, `status`, `region`


**first convert csv file to parquet file**

In [ ]:
# 1. Convert the Spark DataFrame to a local Pandas DataFrame
pandas_df = df.toPandas()

# 2. Save it directly as a Parquet file using Python's native engines
pandas_df.to_parquet("data/input/source.parquet", index=False)

print("Conversion complete via Pandas!")


Now If I run the following query:.

In [9]:
df = spark.read.parquet("data/input/source.parquet")

result = df.filter(df["category"] == "Electronics")

result.show()


+--------+----------+------------+-----------+--------+--------+----------------+------+---------+-------------+----------+
|order_id|product_id|product_name|   category|   price|quantity|discount_percent|region|   status|customer_name|order_date|
+--------+----------+------------+-----------+--------+--------+----------------+------+---------+-------------+----------+
|  100001|      1001|      Laptop|Electronics|74180.89|     4.0|              20| North| Returned|  Customer_17|31-01-2025|
|  100002|      NULL|     Monitor|Electronics|23342.82|    10.0|              20|  West|  Pending| Customer_230|12-10-2025|
|  100003|      1003|    Keyboard|Electronics|69844.13|     6.0|              10| North|Completed| Customer_195|10-04-2025|
|  100014|      1014|      Tablet|Electronics|87884.32|     5.0|              20| North|Completed| Customer_375|13-05-2026|
|  100015|      1015|     Speaker|Electronics|76606.86|     3.0|               5|  East| Returned| Customer_494|05-08-2025|
|  10001


Since the file is stored in **Parquet** format, Spark pushes the filter condition (`category = "Electronics"`) to the Parquet file. Only the matching rows are read into memory, while the remaining rows are skipped.

If the same data were stored in a CSV file, Spark would first read the entire file into memory and then apply the filter, which is slower and uses more memory

# Q10. Write a code snippet to add a new column `final_price` which is the `base_price` multiplied by **1.18** (18% tax).

> **Note:** The question refers to a `base_price` column. However, my dataset contains a `price` column instead, so I have used `price` as the base price to calculate the `final_price`.

In [10]:
# Add a new column 'final_price' by applying 18% tax
df = df.withColumn("final_price", col("price") * 1.18)

# Display the updated DataFrame
df.select("product_name", "price", "final_price").show()

+----------------+--------+------------------+
|    product_name|   price|       final_price|
+----------------+--------+------------------+
|          Laptop|74180.89| 87533.45019999999|
|         Monitor|23342.82|27544.527599999998|
|        Keyboard|69844.13| 82416.07340000001|
|          Jacket|60412.23|        71286.4314|
|        Football|70486.73| 83174.34139999999|
|           Sugar|16349.14|        19291.9852|
|            Eggs|98953.38|       116764.9884|
|            Eggs| 6712.24|         7920.4432|
|            Sofa| 74524.4| 87938.79199999999|
|Badminton Racket| 9181.85|         10834.583|
|Badminton Racket|52958.52| 62491.05359999999|
|            Milk| 45427.0|          53603.86|
|        Football|15368.64|18134.995199999998|
|          Tablet|87884.32|       103703.4976|
|         Speaker|76606.86| 90396.09479999999|
|          Gloves|69017.78|        81440.9804|
|           Table|33874.75|39972.204999999994|
|          Tablet| 7178.65| 8470.806999999999|
|           B

# Q11. What is the difference between Transformations and Actions? Provide two examples of each.

## Transformations vs Actions

In Apache Spark, operations are divided into **Transformations** and **Actions**.

- **Transformations** create a new DataFrame or RDD from an existing one. They are **lazy**, meaning Spark does not execute them immediately. Instead, Spark records them in the DAG and waits until an action is called.

- **Actions** trigger the execution of all pending transformations and return a result or write data to storage.

## Difference Between Transformations and Actions

| Transformation | Action |
|----------------|--------|
| Creates a new DataFrame or RDD. | Executes the transformations. |
| Uses Lazy Evaluation. | Triggers execution immediately. |
| Does not return the final result. | Returns the result or writes output. |

## Examples of Transformations

In [11]:
# Example 1: Filter rows
transformedDF= df.filter(df["price"] > 1000)

# Example 2: Select specific columns
transformedDF.select("product_name", "price")


DataFrame[product_name: string, price: double]

Both `filter()` and `select()` are **Transformations** because they create a new DataFrame but do not execute immediately.

## Examples of Actions

In [12]:
# Example 1: Display the DataFrame
transformedDF.show()

# Example 2: Count the number of rows
transformedDF.count()

+--------+----------+----------------+-----------+--------+--------+----------------+------+---------+-------------+----------+------------------+
|order_id|product_id|    product_name|   category|   price|quantity|discount_percent|region|   status|customer_name|order_date|       final_price|
+--------+----------+----------------+-----------+--------+--------+----------------+------+---------+-------------+----------+------------------+
|  100001|      1001|          Laptop|Electronics|74180.89|     4.0|              20| North| Returned|  Customer_17|31-01-2025| 87533.45019999999|
|  100002|      NULL|         Monitor|Electronics|23342.82|    10.0|              20|  West|  Pending| Customer_230|12-10-2025|27544.527599999998|
|  100003|      1003|        Keyboard|Electronics|69844.13|     6.0|              10| North|Completed| Customer_195|10-04-2025| 82416.07340000001|
|  100004|      1004|          Jacket|   Clothing|60412.23|     1.0|              15| North|Cancelled| Customer_425|06

4842

Both `show()` and `count()` are **Actions** because they trigger the execution of all pending transformations.

# Q12. Write the Spark command to load a Parquet file from `"path/to/input"`, filter out any rows where `user_id` is null, and save the result as a CSV at `"path/to/output"`.

> **Note:** The question refers to a `user_id` column. However, my dataset does not contain this column, so I have used `product_id` as the identifier while implementing the same logic.

In [13]:

df = spark.read.parquet("data/input/source.parquet")

filtered_df = df.filter(col("product_id").isNotNull())

filtered_df.write.mode("overwrite") \
    .option("header", True) \
    .csv("data/output")
filtered_df.count()

4996

# Q13. In Spark Architecture, what is the difference between Client Mode and Cluster Mode?

## Client Mode vs Cluster Mode

Apache Spark applications can run in **Client Mode** or **Cluster Mode**. The main difference between them is the location where the **Driver** program runs.

- In **Client Mode**, the Driver runs on the machine from which the application is submitted.
- In **Cluster Mode**, the Driver runs inside the cluster on one of the worker nodes.

## Difference Between Client Mode and Cluster Mode

| Client Mode | Cluster Mode |
|--------------|--------------|
| Driver runs on the client machine. | Driver runs inside the cluster. |
| Client machine must remain connected until the job finishes. | Client can disconnect after submitting the job. |
| Suitable for development, testing, and debugging. | Suitable for production and long-running jobs. |
| If the client machine fails, the application stops. | If configured, the cluster can recover and continue execution. |

## Example

### Client Mode

Suppose I run the following command from my laptop:

```bash
spark-submit --master yarn my_app.py
```

In **Client Mode**, the **Driver** runs on my laptop, while the **Executors** run on the cluster. If I shut down my laptop or lose the connection, the Spark application stops.

### Cluster Mode

```bash
spark-submit --master yarn --deploy-mode cluster my_app.py
```

In **Cluster Mode**, both the **Driver** and **Executors** run inside the cluster. After submitting the job, I can close my laptop because the cluster continues executing the application independently.

# Q14. Write a query to filter a dataset for rows where the `region` is **'North'** OR the `priority` is **'High'**.

> **Note:** The question refers to a `priority` column. However, my dataset does not contain this column, so I have used the `status` column instead to demonstrate the required filtering operation.

In [15]:
# Filter rows where region is 'North' OR status is 'Completed'

result = df.filter(
    (col("region") == "North") |
    (col("status") == "Completed")
)

result.show()

+--------+----------+----------------+-----------+--------+--------+----------------+------+---------+-------------+----------+
|order_id|product_id|    product_name|   category|   price|quantity|discount_percent|region|   status|customer_name|order_date|
+--------+----------+----------------+-----------+--------+--------+----------------+------+---------+-------------+----------+
|  100001|      1001|          Laptop|Electronics|74180.89|     4.0|              20| North| Returned|  Customer_17|31-01-2025|
|  100003|      1003|        Keyboard|Electronics|69844.13|     6.0|              10| North|Completed| Customer_195|10-04-2025|
|  100004|      1004|          Jacket|   Clothing|60412.23|     1.0|              15| North|Cancelled| Customer_425|06-01-2026|
|  100005|      1005|        Football|     Sports|70486.73|     1.0|               0| South|Completed| Customer_195|12-10-2025|
|  100007|      1007|            Eggs|  Groceries|98953.38|     9.0|               0| South|Completed| C

# Q15. When exploring a dataset, why is it safer to use `.show(5)` instead of `.collect()` on a multi-terabyte dataset?

## Difference Between `.show(5)` and `.collect()`

When working with very large datasets in Apache Spark, it is safer to use `.show(5)` because it displays only the first **5 rows** of the DataFrame.

On the other hand, `.collect()` retrieves **all the rows** from the Spark cluster and brings them to the Driver program. For a multi-terabyte dataset, this can consume a huge amount of memory and may cause the application to fail.

## Comparison

| `.show(5)` | `.collect()` |
|-------------|--------------|
| Displays only the first 5 rows. | Retrieves all rows from the DataFrame. |
| Uses very little memory. | Can consume a large amount of Driver memory. |
| Safe for exploring large datasets. | Risky for very large datasets. |
| Mainly used for previewing data. | Used when all data is required in the Driver program. |

## Example

```python
# Display only the first 5 rows
df.show(5)

# Retrieve all rows to the Driver
data = df.collect()
```

If the DataFrame contains millions or billions of rows, `show(5)` displays only a small sample, while `collect()` attempts to load the entire dataset into the Driver's memory. This can lead to **OutOfMemoryError** or significantly slow down the application.

Conversion complete via Pandas!
